# Betting backtest: modell mot bookmaker

Notebook-version av `src/backtest.py`. Hämtar matcher **och** bookmaker-odds från football-data.co.uk (`src/football_data.py`), tränar om den logistiska modellen inför varje säsong och settlar simulerade 1-enhetsvad mot Pinnacle closing.

Prognosmått som log loss säger om modellen gissar bra. De kan inte säga om den tjänar pengar, för det beror på priset. Den här notebooken mäter båda.

Upplägget:

- En säsong i taget. Modellen för säsong *S* tränas bara på säsonger före *S*.
- Lagmodellens två rader per match slås ihop till en H/D/A-fördelning.
- Ett vad läggs när modellens förväntade värde vid det faktiska priset klarar tröskeln.
- Kontroll: slumpmässigt utfall per match, alltså bookmakerns marginal. En strategi som inte slår den är sämre än ingen modell alls.

Standard här är **Premier League**. Sätt `DIVISIONS = None` för Big 5 som i `reports/backtest`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

here = Path.cwd().resolve()
root = next(path for path in [here, *here.parents] if (path / "src" / "football_data.py").exists())
sys.path.insert(0, str(root))

from src import backtest
from src.feature_engineering import engineer_features
from src.football_data import load_matches
from src.team_dataset import build_team_match_dataset
from src.team_model import BEST_LOGISTIC

# None = alla fem ligor (E0 D1 SP1 I1 F1). ["E0"] = bara Premier League.
DIVISIONS = ["E0"]
FIRST_TEST_SEASON = "2013/14"
EDGE_THRESHOLD = 0.05
OUTCOMES = backtest.OUTCOMES

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

print("pandas:", pd.__version__)
print("Modell:", BEST_LOGISTIC)
print("Closing-odds:", backtest.CLOSING_ODDS)
print("Divisioner:", DIVISIONS or "Big 5")
print("Första testsäsong:", FIRST_TEST_SEASON)

Modell: {'feature_set': 'form_strength_model', 'C': 0.03, 'class_weight': None}
Closing-odds: ['PSCH', 'PSCD', 'PSCA']
Divisioner: ['E0']
Första testsäsong: 2013/14


## 1. Hämta matcher och odds

`load_matches` går mot football-data.co.uk, inte spegeln bakom `data_loader` (som strippar odds). Färdiga säsonger cachas under `data/football-data/` (gitignorerad). Pågående säsong uppdateras var 12:e timme.

In [3]:
raw = load_matches(divisions=DIVISIONS)
raw["MatchDate"] = pd.to_datetime(raw["MatchDate"])
if raw.empty:
    raise ValueError("Inga matcher laddades. Kör första cellen igen efter att ha sparat om src/football_data.py.")

closing = backtest.CLOSING_ODDS
has_closing = raw[closing].notna().all(axis=1) & (raw[closing] > 1).all(axis=1)
start, end = raw.MatchDate.min(), raw.MatchDate.max()

print(f"Matcher: {len(raw):,}")
print(f"Ligor: {', '.join(sorted(raw.League.astype(str).unique()))}")
print(f"Säsonger: {raw.Season.nunique()} ({raw.Season.min()} → {raw.Season.max()})")
print(f"Period: {start:%Y-%m-%d} – {end:%Y-%m-%d}")
print(f"Med Pinnacle closing: {int(has_closing.sum()):,} ({100 * has_closing.mean():.1f}%)")
raw.loc[has_closing, ["Season", "League", "MatchDate", "HomeTeam", "AwayTeam", "FullTimeResult", *closing]].head()

Matcher: 0
Ligor: 
Säsonger: 0 (nan → nan)


ValueError: NaTType does not support strftime

## 2. Pre-match-features och lagdata

Featuremotorn går kronologiskt över hela historiken. Resultat från en match får uppdatera laghistoriken först efter att dagens pre-match-rader är skapade. Varje match blir två lagrader (`Hemma` / `Borta`) med `Target` = Vinst / Oavgjort / Förlust.

In [ ]:
print("Bygger features…")
engineered = engineer_features(raw)
team_data = build_team_match_dataset(raw, engineered)
labeled = team_data.dropna(subset=["Target"])

print(f"Lagrader: {len(team_data):,}")
print(f"Med target: {len(labeled):,}")
print(labeled.Target.value_counts().to_string())
labeled.head()

## 3. Walk-forward: en modell per säsong

Varje testsäsong får en egen `fit_best_logistic` tränad bara på äldre säsonger. Koefficienterna ser alltså aldrig den säsong de predikterar. Features för en match får däremot använda tidigare matcher *i samma säsong* — den informationen finns när vadet läggs.

In [ ]:
predictions = backtest.walk_forward_predictions(team_data, FIRST_TEST_SEASON, verbose=True)
market = backtest.build_market(raw)
joined = predictions.join(market, how="inner")

print(f"\nPredikterade matcher: {len(predictions):,}")
print(f"Med Pinnacle closing: {len(joined):,}")
joined.head()

## 4. Prognoskvalitet på identiska matcher

Samma matcher, tre prognoser: modellen, Pinnacles closing (deviggad) och konstanta träningspriors. Modellen ska slå dummy. Att slå closing-linjen är en annan sak.

In [ ]:
def macro_gini(y, probs, labels=OUTCOMES):
    scores = []
    for index, label in enumerate(labels):
        binary = y.eq(label).astype(int)
        if binary.nunique() < 2:
            continue
        scores.append(2 * roc_auc_score(binary, probs[:, index]) - 1)
    return float(np.mean(scores)) if scores else np.nan


def forecast_row(name, y, probs):
    predicted = np.array(OUTCOMES)[probs.argmax(axis=1)]
    return {
        "källa": name,
        "log_loss": log_loss(y, probs, labels=OUTCOMES),
        "accuracy": accuracy_score(y, predicted),
        "macro_gini": macro_gini(y, probs),
    }


y = joined["result"]
model_probs = joined[OUTCOMES].to_numpy(dtype=float)
market_probs = joined[[f"market_{o}" for o in OUTCOMES]].to_numpy(dtype=float)

train_raw = raw.loc[raw["Season"].astype(str) < FIRST_TEST_SEASON]
priors = (
    train_raw["FullTimeResult"].value_counts(normalize=True)
    .reindex(OUTCOMES).fillna(0.0).to_numpy(dtype=float)
)
dummy_probs = np.tile(priors, (len(joined), 1))

quality = pd.DataFrame([
    forecast_row("Modell", y, model_probs),
    forecast_row("Pinnacle closing", y, market_probs),
    forecast_row("Konstanta träningspriors", y, dummy_probs),
]).set_index("källa")
quality

## 5. Simulerade 1-enhetsvad mot Pinnacle closing

Förväntat värde = `modellens sannolikhet × pris − 1`. Ett vad läggs på varje utfall vars EV klarar tröskeln. Vinnare settlas till `pris − 1`, förlorare till `−1`.

Högre tröskel filtrerar på modellens *egen* sannolikhet. Om modellen är sämre än marknaden plockar filtret ofta de matcher där den har mest fel.

In [ ]:
summary = backtest.summarize(joined)
control = backtest.random_bet_control(joined)
all_outcomes = backtest.simulate(joined, edge_threshold=-np.inf)

print(f"Matcher med prediktion och closing: {len(joined):,}")
print(f"Slumpmässigt utfall per match: {100 * control:+.2f}% ROI")
print(f"Alla utfall på alla matcher: {100 * all_outcomes.profit.mean():+.2f}% ROI")
print("En strategi måste slå slumpkontrollen, inte bara förlora mindre än 100%.")
summary

## 6. Per säsong, utfall och longshots

Samma tröskel som i rapporten (`edge > 0.05`). Longshots är vad där marknadens sannolikhet ligger under 15%.

In [ ]:
bets = backtest.simulate(joined, EDGE_THRESHOLD)

by_season = (
    bets.groupby("Season")
    .agg(bets=("profit", "size"), win_rate=("won", "mean"), roi=("profit", "mean"), profit_units=("profit", "sum"))
)
by_outcome = (
    bets.groupby("outcome")
    .agg(bets=("profit", "size"), win_rate=("won", "mean"), roi=("profit", "mean"), profit_units=("profit", "sum"))
    .reindex(OUTCOMES)
)
longshots = bets.loc[bets.market_prob < 0.15]

print(f"Tröskel {EDGE_THRESHOLD:.2f}: {len(bets):,} vad, ROI {100 * bets.profit.mean():+.2f}%")
print(f"Longshots (marknad < 15%): {len(longshots):,} vad, ROI {100 * longshots.profit.mean():+.2f}%")
print("\nPer utfall:")
display(by_outcome)
print("Per säsong:")
by_season

## 7. Exempel på enskilda vad

Högst förväntat värde enligt modellen, med faktiskt resultat och vinst/förlust. Det här är inte en ranking av *bra* vad — det är där modellen är mest oenig med priset.

In [ ]:
ids = raw.copy()
ids["MatchId"] = (
    ids["League"].astype(str) + "_"
    + pd.to_datetime(ids["MatchDate"]).dt.strftime("%Y%m%d") + "_"
    + ids["HomeTeam"].astype(str) + "_"
    + ids["AwayTeam"].astype(str)
)
lookup = ids.set_index("MatchId")[["MatchDate", "HomeTeam", "AwayTeam"]]

sample = bets.join(lookup, how="left").sort_values("expected_value", ascending=False)
sample[[
    "MatchDate", "HomeTeam", "AwayTeam", "Season", "outcome", "price",
    "model_prob", "market_prob", "expected_value", "profit", "won",
]].head(15)